# Rotten Fruit Detector — SageMaker JumpStart EfficientDet D1

Fine-tunes **SSD EfficientDet D1** (`tensorflow-od1-ssd-efficientdet-d1-640x640-coco17-tpu-8`) via **SageMaker JumpStart**. AWS ships a pre-built training container with the TF Object Detection API (protobuf / pycocotools / `object_detection`) already resolved — this notebook only *orchestrates* the job, so there are **no dependency errors to fix locally**.

Flow: convert the Roboflow CSV export → JumpStart format → upload to S3 → `.fit()` launches an ephemeral GPU training instance → deploy an endpoint → predict.

Expected `dataset.zip` (Roboflow **TensorFlow Object Detection** CSV export):

```text
train/_annotations.csv + *.jpg
valid/_annotations.csv + *.jpg
test/_annotations.csv  + *.jpg
```
Run on any small kernel (e.g. `ml.t3.medium`) — the heavy GPU work happens on the separate training instance, not this notebook.

In [ ]:
# 1. Setup: SageMaker session, role, bucket
%pip install -q -r ../requirements.txt

import sagemaker
from pathlib import Path

session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = session.default_bucket()
PROJECT_DIR = Path.cwd().parent
print("Role:", role)
print("Bucket:", bucket)

In [ ]:
# 2. Get the raw dataset from S3 and unzip it
import zipfile
import boto3

S3_DATA_ZIP = "s3://YOUR_BUCKET/datasets/dataset.zip"  # your zipped Roboflow CSV export

b, key = S3_DATA_ZIP.replace("s3://", "").split("/", 1)
zip_path = PROJECT_DIR / "dataset.zip"
boto3.client("s3").download_file(b, key, str(zip_path))

EXTRACT_DIR = PROJECT_DIR / "dataset"
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(EXTRACT_DIR)

def find_dataset_root(base):
    base = Path(base)
    for path in [base, *(p for p in base.rglob("*") if p.is_dir())]:
        if (path / "train").is_dir():
            return path
    raise FileNotFoundError(f"No 'train' folder found under {base}")

DATA_DIR = find_dataset_root(EXTRACT_DIR)
print("Dataset root:", DATA_DIR)

In [ ]:
# 3. Convert Roboflow CSV -> JumpStart format (images/ + annotations.json)
import sys
sys.path.insert(0, str(PROJECT_DIR / "src"))

from dataset import prepare_jumpstart_dataset

# Use train split for fine-tuning; add "valid" to splits to include more data.
jumpstart_dir, class_names = prepare_jumpstart_dataset(
    DATA_DIR, PROJECT_DIR / "jumpstart_data", splits=("train",)
)

In [ ]:
# 4. Upload the JumpStart input directory to S3 (trailing slash required)
training_s3_uri = session.upload_data(
    path=str(jumpstart_dir), bucket=bucket, key_prefix="rotten-fruit/jumpstart-input"
)
if not training_s3_uri.endswith("/"):
    training_s3_uri += "/"
print("Training data:", training_s3_uri)

In [ ]:
# 5. Fine-tune EfficientDet D1 on an ephemeral GPU instance.
#    train() downloads AWS's JumpStart training script, patches its train-step @tf.function
#    with reduce_retracing=True (AWS's script leaks host RAM via per-step retracing on
#    variable box counts -> SIGKILL), and runs it via a plain Estimator.
#    ml.g5.2xlarge (A10G 24GB GPU, 32GB RAM). Needs its 'for training job usage' quota >= 1.
from train import train

estimator = train(
    training_s3_uri=training_s3_uri,
    role=role,
    output_path=f"s3://{bucket}/rotten-fruit/output",
    instance_type="ml.g5.2xlarge",
    epochs=5,
)

In [ ]:
# 6. Deploy an endpoint and run a prediction, then clean up
from train import deploy
from predict import predict_image

predictor = deploy(estimator, instance_type="ml.m5.xlarge")

test_image = sorted((DATA_DIR / "test").glob("*.jpg"))[0]
for det in predict_image(predictor, test_image, class_names=class_names)[:10]:
    print(det)

# IMPORTANT: delete the endpoint so it stops incurring charges.
predictor.delete_endpoint()